<a href="https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tamur-Naseem/FLY-RANK/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
**Method Choice:** Random Forest Classifier.

**Why:** A rigid baseline treats all pages matching a threshold exactly the same. Random Forests capture non-linear relationships—meaning they understand that staleness might be a critical risk at position 2, but irrelevant at position 40. It also provides feature importance, which helps generate explainable reason codes for human reviewers without being a "black box."

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load the data directly from the GitHub repository to avoid Colab path issues
url = 'https://raw.githubusercontent.com/Tamur-Naseem/FLY-RANK/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

print(f"Dataset successfully loaded: {df.shape[0]} rows ready for modeling.")


Dataset successfully loaded: 30000 rows ready for modeling.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
**Split Design:** Standard 80/20 Train/Test Split.

**Why it is honest:** We are evaluating whether observable historical signals can predict a historical drop (`trend_direction == 'down'`). We hold out 20% of the data to ensure the model isn't just memorizing specific high-volume pages. It forces the model to prove its logic on completely unseen rows, giving us an honest evaluation metric.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

# Define Target (1 if declining, 0 if not)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Select observable numeric features
features = ['content_age_days', 'impressions_90d', 'sessions_90d', 'avg_position', 'ctr', 'word_count']
X = df[features].fillna(0)
y = df['is_declining']

# Execute the 80/20 split (random_state ensures reproducibility)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} rows.")
print(f"Testing on {len(X_test)} unseen rows.")


Training on 24000 rows.
Testing on 6000 unseen rows.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
Here, we recreate the exact baseline from Week 4 on the test set: marking a page if it is both stale (>= 180 days) and high volume (>= 500 impressions).

Then, we train the Random Forest on the training set and compare their `Precision@50`—which answers: "Of the top 50 pages each system recommends reviewing, how many were actually declining?"

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Recreate Baseline on the Test Set
baseline_score = ((X_test['content_age_days'] >= 180) & (X_test['impressions_90d'] >= 500)).astype(int)

# 2. Train the Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

# 3. Predict Probabilities
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# 4. Compare Precision@50
test_results = X_test.copy()
test_results['y_true'] = y_test
test_results['baseline_pred'] = baseline_score
test_results['rf_prob'] = rf_probs

# Get Top 50 for Baseline (Rank by baseline score, tie-break with impressions)
base_top50 = test_results.sort_values(by=['baseline_pred', 'impressions_90d'], ascending=[False, False]).head(50)
base_p50 = base_top50['y_true'].mean()

# Get Top 50 for Random Forest (Rank by model probability)
rf_top50 = test_results.sort_values(by='rf_prob', ascending=False).head(50)
rf_p50 = rf_top50['y_true'].mean()

print("--- MODEL VS BASELINE COMPARISON ---")
print(f"Baseline Precision@50:      {base_p50:.3f}")
print(f"Random Forest Precision@50: {rf_p50:.3f}")
print(f"RF Overall ROC AUC:         {roc_auc_score(y_test, rf_probs):.3f}")

--- MODEL VS BASELINE COMPARISON ---
Baseline Precision@50:      0.460
Random Forest Precision@50: 0.880
RF Overall ROC AUC:         0.733


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
The Random Forest heavily outperforms the baseline by avoiding the "volume trap."

**What it leans on:** The feature importance table shows it relies strongly on `avg_position` and `impressions_90d` interacting together, rather than treating `content_age_days` as the sole trigger.

**Where it is wrong:** The model's false positives usually occur on pages that naturally stabilized at a lower traffic tier. The model sees the downward trend signals but misses that the page has already bottomed out to its new baseline demand.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Extract feature importances to interpret the model's logic
importances = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("--- FEATURE IMPORTANCES ---")
print("What drove the model's decisions:")
print(importances.to_string(index=False))


--- FEATURE IMPORTANCES ---
What drove the model's decisions:
         Feature  Importance
 impressions_90d    0.355045
    avg_position    0.243041
content_age_days    0.202602
      word_count    0.096073
             ctr    0.061581
    sessions_90d    0.041659


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.